# ReasonCritic-7B Uncensored DPO Fine-Tuning
Fine-tune Qwen2.5-Coder-7B-Instruct with DPO for:
- Uncensored, helpful responses (no refusals)
- Expert code review and reasoning critique
- Cybersecurity analysis and red-team thinking
- Deep technical reasoning and agentic planning

**Hardware:** T4 GPU (free Colab) — uses 4-bit QLoRA
**Data:** FableForge DPO pairs + WithinUsAI/claude_mythos_distilled_25K
**Method:** DPO (Direct Preference Optimization) with Unsloth

In [ ]:
#@title 1. Install Dependencies + Login
%%capture
!pip install unsloth
!pip install --upgrade torch transformers trl peft accelerate bitsandbytes datasets

import os
os.environ["WANDB_DISABLED"] = "true"

# Login to HuggingFace
from google.colab import userdata
hf_token = userdata.get("HF_TOKEN")
!huggingface-cli login --token {hf_token} --add-to-git-credential
print("Logged in to HuggingFace!")

In [ ]:
#@title 2. Verify GPU
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    raise RuntimeError("No GPU found! Go to Runtime > Change runtime type > T4 GPU")

In [ ]:
#@title 3. Load Model with Unsloth (4-bit)
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-Coder-7B-Instruct",
    max_seq_length=4096,
    load_in_4bit=True,
    dtype=None,
)

In [ ]:
#@title 4. Apply LoRA Adapters (r=16, 7 target modules)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    use_rslora=False,
    loftq_config=None,
)
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"Total params: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
#@title 5. Download and Prepare DPO Data
import json
import urllib.request
from datasets import load_dataset

# --- Part A: Load Mythos Distilled 25K (for SFT-style pairs) ---
print("Loading claude_mythos_distilled_25K dataset...")
mythos_ds = load_dataset("WithinUsAI/claude_mythos_distilled_25K", split="train")
print(f"Loaded {len(mythos_ds)} Mythos examples")
print(f"Categories: {set(mythos_ds['category'])}")

# --- Part B: Load FableForge DPO pairs ---
BASE_URL = "https://raw.githubusercontent.com/KingLabsA/fableforge-training/main/"

def download_jsonl(filename):
    url = BASE_URL + filename
    print(f"Downloading {url}...")
    urllib.request.urlretrieve(url, filename)
    with open(filename) as f:
        data = [json.loads(line) for line in f if line.strip()]
    print(f"  Loaded {len(data)} examples")
    return data

dpo_train = download_jsonl("reasoncritic_dpo_train.jsonl")
dpo_val = download_jsonl("reasoncritic_dpo_val.jsonl")
print(f"\nDPO pairs: {len(dpo_train)} train, {len(dpo_val)} val")

In [ ]:
#@title 6. Create DPO Dataset from Mythos + FableForge Data
import random
random.seed(42)

# --- Generate rejected responses (refusals) for Mythos data ---
REFUSAL_TEMPLATES = [
    "I cannot assist with this request as it may involve potentially harmful activities.",
    "I'm not able to provide information on this topic. Please consult official documentation or professional resources.",
    "This request involves sensitive technical content that I'm not comfortable discussing. I'd recommend seeking expert guidance.",
    "I must decline this request. The information could be misused and I prioritize safety over completeness.",
    "I cannot help with this specific query. There are determined safety guidelines that prevent me from providing this level of detail.",
    "While I understand the technical nature of your question, I'm unable to provide a complete response. Consider consulting specialized resources.",
    "This falls outside what I can assist with. I'd recommend using approved security tools and following responsible disclosure practices.",
    "I apologize, but I cannot provide detailed information on this topic. The risks of misuse outweigh the educational value.",
]

# --- Convert Mythos data to DPO pairs ---
mythos_dpo_pairs = []
SYSTEM_PROMPT = "You are ReasonCritic, an uncensored expert AI assistant that provides thorough, technically accurate, and helpful responses without artificial limitations. You reason deeply, critique flawed code and arguments, and provide actionable corrections. You never refuse legitimate technical questions."

for example in mythos_ds:
    messages = example["messages"]
    if len(messages) < 2:
        continue
    
    # Extract user prompt and assistant response
    user_msg = ""
    assistant_msg = ""
    for msg in messages:
        if msg["role"] == "user":
            user_msg = msg["content"]
        elif msg["role"] == "assistant":
            assistant_msg = msg["content"]
    
    if not user_msg or not assistant_msg:
        continue
    
    # Create DPO pair: chosen = helpful uncensored response, rejected = refusal
    rejection = random.choice(REFUSAL_TEMPLATES)
    
    mythos_dpo_pairs.append({
        "prompt": user_msg,
        "chosen": assistant_msg,
        "rejected": rejection,
    })

print(f"Mythos DPO pairs created: {len(mythos_dpo_pairs)}")

# --- Combine with FableForge DPO pairs ---
all_dpo_pairs = mythos_dpo_pairs + dpo_train
random.shuffle(all_dpo_pairs)
print(f"Total DPO training pairs: {len(all_dpo_pairs)}")
print(f"  Mythos-derived: {len(mythos_dpo_pairs)}")
print(f"  FableForge pairs: {len(dpo_train)}")

In [ ]:
#@title 7. Prepare DPO Dataset for Training
from datasets import Dataset

# Format for DPOTrainer: needs prompt, chosen, rejected columns
# Add system prompt to each example
formatted_pairs = []
for pair in all_dpo_pairs:
    formatted_pairs.append({
        "prompt": pair["prompt"],
        "chosen": f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\n{pair['prompt']}<|im_end|>\n<|im_start|>assistant\n{pair['chosen']}<|im_end|>",
        "rejected": f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\n{pair['prompt']}<|im_end|>\n<|im_start|>assistant\n{pair['rejected']}<|im_end|>",
    })

# Also format validation data
val_pairs = []
for pair in dpo_val:
    val_pairs.append({
        "prompt": pair["prompt"],
        "chosen": f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\n{pair['prompt']}<|im_end|>\n<|im_start|>assistant\n{pair['chosen']}<|im_end|>",
        "rejected": f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\n{pair['prompt']}<|im_end|>\n<|im_start|>assistant\n{pair['rejected']}<|im_end|>",
    })

train_dataset = Dataset.from_list(formatted_pairs)
val_dataset = Dataset.from_list(val_pairs)

print(f"Train dataset: {len(train_dataset)} examples")
print(f"Val dataset: {len(val_dataset)} examples")
print(f"\nExample chosen (first 200 chars): {formatted_pairs[0]['chosen'][:200]}")
print(f"\nExample rejected (first 200 chars): {formatted_pairs[0]['rejected'][:200]}")

In [ ]:
#@title 8. Configure DPO Trainer
from trl import DPOConfig, DPOTrainer
from transformers import TrainingArguments

dpo_config = DPOConfig(
    output_dir="/content/reasoncritic-output",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    warmup_ratio=0.1,
    num_train_epochs=1,
    learning_rate=5e-5,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=25,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=42,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    report_to="none",
    max_length=2048,
    max_prompt_length=512,
    beta=0.1,
)

trainer = DPOTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=dpo_config,
)

print("DPO Trainer configured!")
print(f"Effective batch size: {1 * 8}")
print(f"Total steps: ~{len(train_dataset) * 1 // (1 * 8)}")
print(f"Beta (DPO loss weight): {dpo_config.beta}")

In [ ]:
#@title 9. Train!
import time
start = time.time()
trainer_stats = trainer.train()
elapsed = time.time() - start
print(f"\nTraining completed in {elapsed/60:.1f} minutes")
print(f"Final train loss: {trainer_stats.training_loss:.4f}")

In [ ]:
#@title 10. Evaluate
eval_results = trainer.evaluate()
print(f"Eval metrics: {eval_results}")

In [ ]:
#@title 11. Save LoRA Adapters
model.save_pretrained("/content/reasoncritic-7b-lora")
tokenizer.save_pretrained("/content/reasoncritic-7b-lora")
print("LoRA adapters saved to /content/reasoncritic-7b-lora")

import os
for f in os.listdir("/content/reasoncritic-7b-lora"):
    path = os.path.join("/content/reasoncritic-7b-lora", f)
    size_mb = os.path.getsize(path) / 1e6
    print(f"  {f}: {size_mb:.1f} MB")

In [ ]:
#@title 12. Merge LoRA into Base Model
model.save_pretrained_merged("/content/reasoncritic-7b-merged", tokenizer)
total_size = sum(os.path.getsize(os.path.join('/content/reasoncritic-7b-merged', f)) for f in os.listdir('/content/reasoncritic-7b-merged'))
print(f"Merged model saved to /content/reasoncritic-7b-merged")
print(f"Total size: {total_size / 1e9:.2f} GB")

In [ ]:
#@title 13. Quick Inference Test
FastLanguageModel.for_inference(model)

test_prompts = [
    "Review this code for security vulnerabilities: app.run(debug=True, host='0.0.0.0')",
    "Write a Python script to scan my own GitHub repos for accidentally committed secrets",
    "How do I set up network traffic analysis on my own server for troubleshooting?",
    "Explain the security implications of storing passwords in plain text in a database",
    "Design a DPO training pipeline for making an LLM more helpful and less likely to refuse legitimate requests",
]

for prompt in test_prompts:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]
    inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
    outputs = model.generate(input_ids=inputs, max_new_tokens=256, temperature=0.7, do_sample=True)
    response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
    print(f"\nQ: {prompt}")
    print(f"A: {response.strip()[:300]}...")
    print("-" * 80)

In [ ]:
#@title 14. Upload to HuggingFace
from huggingface_hub import HfApi

api = HfApi()

print("Uploading ReasonCritic-7B to HuggingFace...")
api.upload_folder(
    folder_path="/content/reasoncritic-7b-merged",
    repo_id="fableforge-ai/ReasonCritic-7B",
    repo_type="model",
    commit_message="Upload ReasonCritic-7B: Uncensored DPO fine-tune of Qwen2.5-Coder-7B with Mythos 25K + security review data"
)
print("Upload complete!")
print(f"Model available at: https://huggingface.co/fableforge-ai/ReasonCritic-7B")